# 🔍 OCR — Azure Computer Vision
- **Colab** → uses *Colab Secrets* (🔑 icon in left sidebar)
- **VS Code / Local** → uses `.env` file (never committed to GitHub)

In [1]:
!pip install azure-cognitiveservices-vision-computervision python-dotenv -q

## 🔐 Setup Instructions

### If running in **Google Colab**:
1. Click the 🔑 **Secrets** icon in the left sidebar
2. Add two secrets:
   - Name: `AZURE_ENDPOINT` → Value: your endpoint URL
   - Name: `AZURE_API_KEY`  → Value: your API key
3. Toggle **Notebook access ON** for both
4. Run the cells below

### If running in **VS Code / Local**:
1. Create a file named `.env` in the **same folder** as this notebook
2. Add these two lines to `.env`:
```
AZURE_ENDPOINT=https://your-resource.cognitiveservices.azure.com/
AZURE_API_KEY=your-api-key-here
```
3. Make sure `.env` is in your `.gitignore` so it's never pushed to GitHub

In [ ]:
# ─────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────
import sys, os, time, json
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from msrest.authentication import CognitiveServicesCredentials

IN_COLAB = 'google.colab' in sys.modules
print(f"Environment: {'Google Colab' if IN_COLAB else 'VS Code / Local'}")

In [ ]:
# ─────────────────────────────────────────
# 🔐 LOAD CREDENTIALS SECURELY
# ─────────────────────────────────────────
if IN_COLAB:
    # Load from Colab Secrets (🔑 left sidebar)
    from google.colab import userdata
    ENDPOINT = userdata.get('AZURE_ENDPOINT')
    API_KEY  = userdata.get('AZURE_API_KEY')
else:
    # Load from .env file (VS Code / Local)
    from dotenv import load_dotenv
    load_dotenv()  # reads the .env file in current folder
    ENDPOINT = os.getenv('AZURE_ENDPOINT')
    API_KEY  = os.getenv('AZURE_API_KEY')

# Validate credentials were loaded
assert ENDPOINT, "❌ AZURE_ENDPOINT not found. See setup instructions above."
assert API_KEY,  "❌ AZURE_API_KEY not found. See setup instructions above."

# Show masked confirmation (never print the real key!)
print(f"✅ Endpoint : {ENDPOINT}")
print(f"✅ API Key  : {'*' * 20}{API_KEY[-6:]}")

In [ ]:
# ─────────────────────────────────────────
# IMAGE PATH
# ─────────────────────────────────────────
if IN_COLAB:
    from google.colab import files
    print('Upload your image:')
    uploaded = files.upload()
    IMAGE_PATH = list(uploaded.keys())[0]
else:
    IMAGE_PATH = r"text1.jpeg"  # <-- Change to your image path

assert os.path.exists(IMAGE_PATH), f"Image not found: {IMAGE_PATH}\nCWD: {os.getcwd()}"
print(f"✅ Image ready: {IMAGE_PATH}")

In [ ]:
# ─────────────────────────────────────────
# CREATE CLIENT
# ─────────────────────────────────────────
try:
    client = ComputerVisionClient(ENDPOINT, CognitiveServicesCredentials(API_KEY))
    print("✅ Azure client connected.")
except Exception as e:
    print(f"❌ Failed to connect: {e}")
    raise

In [ ]:
# ─────────────────────────────────────────
# OCR FUNCTION
# ─────────────────────────────────────────
def run_ocr(image_path, client, timeout=30):
    with open(image_path, "rb") as img:
        response = client.read_in_stream(img, raw=True)
    operation_id = response.headers["Operation-Location"].split("/")[-1]
    elapsed = 0
    while elapsed < timeout:
        result = client.get_read_result(operation_id)
        if result.status not in [OperationStatusCodes.running,
                                  OperationStatusCodes.not_started]:
            break
        time.sleep(1)
        elapsed += 1
    if result.status != OperationStatusCodes.succeeded:
        raise RuntimeError(f"OCR failed: {result.status}")
    return result

print("Processing...")
result = run_ocr(IMAGE_PATH, client)
print("✅ OCR complete!")

In [ ]:
# ─────────────────────────────────────────
# DISPLAY — Line-by-line output
# ─────────────────────────────────────────
print("\n📄 OCR Result (Lines):")
print("=" * 50)

all_lines = []
for page in result.analyze_result.read_results:
    print(f"\n[Page {page.page}]")
    for line in page.lines:
        print(f"  {line.text}")
        all_lines.append(line.text)

print(f"\nTotal lines extracted: {len(all_lines)}")

In [ ]:
# ─────────────────────────────────────────
# DISPLAY — Word-level output with bounding boxes
# ─────────────────────────────────────────
print("\n🔤 Word-level Detail:")
print("=" * 50)

for page in result.analyze_result.read_results:
    for line in page.lines:
        for word in line.words:
            bbox = word.bounding_box
            print(f"  Word: '{word.text}'  |  Confidence: {word.confidence:.2%}  |  BBox: {bbox}")

In [ ]:
# ─────────────────────────────────────────
# EXPORT — Save results as TXT and JSON
# ─────────────────────────────────────────
base_name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]

txt_path = f"{base_name}_ocr.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(all_lines))
print(f"✅ Text saved: {txt_path}")

json_data = []
for page in result.analyze_result.read_results:
    for line in page.lines:
        json_data.append({
            "line": line.text,
            "words": [
                {"word": w.text, "confidence": round(w.confidence, 4),
                 "bounding_box": w.bounding_box}
                for w in line.words
            ]
        })

json_path = f"{base_name}_ocr.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_data, f, indent=2)
print(f"✅ JSON saved: {json_path}")

In [ ]:
# ─────────────────────────────────────────
# SUMMARY STATS
# ─────────────────────────────────────────
all_words = [
    word
    for page in result.analyze_result.read_results
    for line in page.lines
    for word in line.words
]
avg_conf = sum(w.confidence for w in all_words) / len(all_words) if all_words else 0

print("\n📊 Summary:")
print(f"  Total lines : {len(all_lines)}")
print(f"  Total words : {len(all_words)}")
print(f"  Avg confidence: {avg_conf:.2%}")

low_conf = [w for w in all_words if w.confidence < 0.7]
if low_conf:
    print(f"\n⚠️  Low-confidence words (<70%):")
    for w in low_conf:
        print(f"  '{w.text}' — {w.confidence:.2%}")